# Loan Default Prediction 

### ingestion

loading the raw data and verifying.

In [76]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("yasserh/loan-default-dataset")
file_path = os.path.join(path, "Loan_Default.csv")
data = pd.read_csv(file_path)

Validating the data

In [77]:
data.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [78]:
data.shape

(148670, 34)

In [79]:
data.columns

Index(['ID', 'year', 'loan_limit', 'Gender', 'approv_in_adv', 'loan_type',
       'loan_purpose', 'Credit_Worthiness', 'open_credit',
       'business_or_commercial', 'loan_amount', 'rate_of_interest',
       'Interest_rate_spread', 'Upfront_charges', 'term', 'Neg_ammortization',
       'interest_only', 'lump_sum_payment', 'property_value',
       'construction_type', 'occupancy_type', 'Secured_by', 'total_units',
       'income', 'credit_type', 'Credit_Score', 'co-applicant_credit_type',
       'age', 'submission_of_application', 'LTV', 'Region', 'Security_Type',
       'Status', 'dtir1'],
      dtype='str')

In [80]:
X = data.drop(columns=['Status','Gender','ID','year'])   
Y = data['Status']

### pre-processing
Handles nulls and encoding

After doing the explanatory data analysis strategy for imputation of NaN values for numerical features which exhibits the skewness by median and for those numerical features, which does not exhibits skewness by median(since both mean and median will be the same, but to ensure consistency throughout the data media will be used for imputation)

For categorical features, the strategy for imputation (after doing the data analysis of the features with other features through cross tabulation and not finding a supporting relationship between them) by mode.


In [81]:
num_cols = [
    "rate_of_interest",
    "Interest_rate_spread",
    "Upfront_charges",
    "term",
    "property_value",
    "income",
    "LTV",
    "dtir1"
]

cat_cols = [
    "loan_limit",
    "approv_in_adv",
    "submission_of_application",
    "age",
    "loan_purpose",
    "Neg_ammortization"
]

# Numerical
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

# Categorical
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])



One-Hot Encoding categorical features.

In [88]:
Cat_cols = [col for col in X.columns if X[col].dtype in ['str', 'category']]

#One-Hot Encoding 

X = pd.get_dummies(data = X,
                         prefix = Cat_cols,
                         columns = Cat_cols)

### Feature Selection



Pearson Correlation Coefficients for Numerical Features

In [89]:
from scipy import stats
Num_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]

num_df = X[Num_cols]
pearson_dic = {i: stats.pearsonr(num_df[i], data['Status']) for i in num_df.columns}
pd.DataFrame(pearson_dic, index = ['Pearson Statistic', 'p-value']).T.sort_values(by='Pearson Statistic', ascending = False)

,Pearson Statistic,p-value
dtir1,0.082432,1.929235e-222
LTV,0.042656,7.788582e-61
Credit_Score,0.004004,1.226544e-01
term,-0.000207,9.363109e-01
loan_amount,-0.036825,8.690628e-46
rate_of_interest,-0.046738,1.119591e-72
Interest_rate_spread,-0.049536,2.031071e-81
income,-0.060618,4.882970e-121
property_value,-0.080905,2.522136e-214
Upfront_charges,-0.095094,1.210288e-295
